# Surface water mass transformation on the Antarctic shelf

Surface water mass transformation (SWMT) driven by surface heat and salt/freshwater fluxes, in $\sigma_1$ density classes,
for the Southern Ocean south of 59°S and for the Antarctic continental shelf (poleward of the 1000 m isobath).
ACCESS-OM3 is compared with ACCESS-OM2 (`025deg_jra55_iaf_omip2_cycle1`) for IAF runs.

The method follows the COSIMA recipe
[Surface_Water_Mass_Transformation.ipynb](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Surface_Water_Mass_Transformation.ipynb)
([Newsom et al. 2016](https://journals.ametsoc.org/doi/full/10.1175/JCLI-D-15-0513.1);
[Abernathey et al. 2016](https://www.nature.com/articles/ngeo2749)):

$$
\Omega(\sigma, t) = \frac{\partial}{\partial\sigma} \iint_{\sigma'<\sigma} \Big(\frac{\partial\sigma}{\partial\theta} \dot\theta + \frac{\partial\sigma}{\partial S}\dot S\Big) \, \mathrm{d}x \, \mathrm{d}y
$$

Positive values mean transformation towards denser classes. Where $\Omega$ decreases with density (on the dense side of its peak),
water converges in density space at the surface and is formed and subducted.

GitHub issue: https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/107

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
# e.g. https://access-om3-configs.access-hive.org.au/latest/Experiments/
# esm_file = "/g/data/ol01/outputs/access-om3-8km/MC_8km_jra_ryf-test2-expt-c349f800/experiment_datastore.json"
# esm_file = "/g/data/ik11/outputs/access-om3-025/MC_25km_era_iaf+wombatlite-test5.1-era-b3d64e3d/datastore.json"
# esm_file = "/g/data/ik11/outputs/access-om3-025/MC_25km_jra_ryf+wombatlite-test5.1-5c6ad028/datastore.json"
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
if not papermill: 
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import os
import warnings

import cartopy.crs as ccrs
import cmocean as cm
import dask
import gsw
import intake
import matplotlib.path as mpath
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from distributed import Client
from scipy import ndimage

warnings.filterwarnings("ignore", category=RuntimeWarning)  # NaN means over land etc.

In [ ]:
client = Client(threads_per_worker=1)
print(client.dashboard_link)

In [ ]:
IAF = esm_file.find('iaf') > 0
IAF

### Open the intake-esm datastore

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

### Settings

For IAF runs, the analysis uses a fixed decade that is in both the OM3 and OM2 experiments (`025deg_jra55_iaf_omip2_cycle1` ends in 2018).
Otherwise it uses the last 10 years of the OM3 run, and the OM2 comparison is skipped: the OM2 RYF run
(`025deg_jra55_ryf9091_gadi`) has no monthly temperature over most of its length.

In [ ]:
lat_max = -59          # northern boundary of the Southern Ocean region (degrees N)
shelf_isobath = 1000   # the shelf is the region poleward of this isobath (m)

# sigma_1 bins (kg m-3 - 1000), as in the COSIMA recipe
isopycnal_bin_edges = np.arange(31, 33.5, 0.02)

c_p = 3992.1  # J kg-1 degC-1, as in the COSIMA recipe

om2_expt = '025deg_jra55_iaf_omip2_cycle1'
do_om2 = IAF

if IAF:
    t_slice = slice('2009', '2018')  # years in both the OM3 and OM2 experiments
else:
    years = np.unique(datastore.search(variable="hfds", frequency="1mon").df.start_date.str[:4])
    t_slice = slice(years[-10], years[-1])
t_slice

## Load the diagnostics

Monthly diagnostics needed for each model (see the COSIMA recipe for details):

| Quantity | ACCESS-OM3 (MOM6) | ACCESS-OM2 (MOM5) |
| --- | --- | --- |
| Surface temperature | `bigthetao` (conservative), or `thetao` (potential) converted with `gsw` | `pot_temp` converted with `gsw` |
| Surface salinity | `so` (practical; the model's absolute salinity is converted to practical for output) | `salt` (practical) |
| Water flux into ocean | `wfo` | `pme_river` |
| Salt flux into ocean | `sfdsi` (sea ice) + `salt_flux_added` (restoring) | `sfc_salt_flux_ice` + `sfc_salt_flux_restore` |
| Net surface heat flux | `hfds` | `sfc_hflux_coupler` + `sfc_hflux_from_runoff` + `frazil_3d_int_z` + `sfc_hflux_pme` |
| Cell area | `areacello` | `area_t` |
| Ocean depth | `deptho` | `ht` |

In [ ]:
TO_DASK_KWARGS = dict(
    xarray_combine_by_coords_kwargs=dict(compat="override", data_vars="minimal", coords="minimal"),
)

def surface_fields(SP, T, z, lat, lon, temp_is_conservative):
    """Absolute salinity, conservative temperature, pressure and sigma_1 at the surface (as in the COSIMA recipe)"""
    pressure = gsw.p_from_z(-z, lat)
    SA = gsw.SA_from_SP(SP, pressure, lon, lat)
    CT = T if temp_is_conservative else gsw.CT_from_pt(SA, T)
    return dict(pressure=pressure, SA=SA, CT=CT, pot_rho_1=gsw.sigma1(SA, CT))


def load_om3():
    def get(variable):
        return datastore.search(variable=variable, frequency="1mon").to_dask(
            xarray_open_kwargs=dict(chunks={"time": 12}, decode_timedelta=True), **TO_DASK_KWARGS
        )[variable].sel(yh=slice(None, lat_max), time=t_slice)

    have_ct = len(datastore.search(variable="bigthetao", frequency="1mon").df) > 0
    T = get("bigthetao" if have_ct else "thetao").isel(z_l=0)
    SP = get("so").isel(z_l=0)
    fluxes = dict(
        wfo=get("wfo"),
        salt_flux=get("sfdsi") + get("salt_flux_added"),
        heat_flux=get("hfds"),
    )
    T, SP, fluxes = dask.compute(T, SP, fluxes)
    xr.align(T, SP, *fluxes.values(), join="exact")  # fail loudly rather than silently dropping months

    static_path = datastore.search(filename="access-om3.mom6.static.nc").df.path.iloc[0]
    static = xr.open_dataset(static_path).sel(yh=slice(None, lat_max))
    lat2d, lon2d = xr.broadcast(T.yh, T.xh)

    ds = dict(fluxes)
    ds.update(surface_fields(SP, T, SP.z_l, lat2d, lon2d, temp_is_conservative=have_ct))
    ds["area"] = static["areacello"].load().assign_coords(xh=T.xh, yh=T.yh)
    ds["maximum_depth"] = static["deptho"].load().assign_coords(xh=T.xh, yh=T.yh)
    ds["days_per_month"] = T.time.dt.days_in_month
    return {k: (v.rename({"xh": "x", "yh": "y"}) if "xh" in v.dims else v) for k, v in ds.items()}


ds_om3 = load_om3()
print(ds_om3["CT"].time.values[[0, -1]])

In [ ]:
def load_om2():
    cat = intake.cat.access_nri[om2_expt]

    def get(variable):
        return cat.search(variable=variable, frequency="1mon").to_dask(
            xarray_open_kwargs=dict(chunks={"time": 12}, decode_timedelta=True), **TO_DASK_KWARGS
        )[variable].sel(yt_ocean=slice(None, lat_max), time=t_slice)

    T = get("pot_temp").isel(st_ocean=0) - 273.15  # K -> degC
    SP = get("salt").isel(st_ocean=0)
    fluxes = dict(
        wfo=get("pme_river"),
        salt_flux=get("sfc_salt_flux_ice") + get("sfc_salt_flux_restore"),
        heat_flux=(get("sfc_hflux_coupler") + get("sfc_hflux_from_runoff")
                   + get("frazil_3d_int_z") + get("sfc_hflux_pme")),
    )
    T, SP, fluxes = dask.compute(T, SP, fluxes)
    xr.align(T, SP, *fluxes.values(), join="exact")  # fail loudly rather than silently dropping months

    grid_path = cat.search(filename="ocean_grid.nc").df.path.iloc[0]
    grid = xr.open_dataset(grid_path).sel(yt_ocean=slice(None, lat_max))
    lat2d, lon2d = xr.broadcast(T.yt_ocean, T.xt_ocean)

    ds = dict(fluxes)
    ds.update(surface_fields(SP, T, SP.st_ocean, lat2d, lon2d, temp_is_conservative=False))
    ds["area"] = grid["area_t"].load().assign_coords(xt_ocean=T.xt_ocean, yt_ocean=T.yt_ocean)
    ds["maximum_depth"] = grid["ht"].load().assign_coords(xt_ocean=T.xt_ocean, yt_ocean=T.yt_ocean)
    ds["days_per_month"] = T.time.dt.days_in_month
    return {k: (v.rename({"xt_ocean": "x", "yt_ocean": "y"}) if "xt_ocean" in v.dims else v) for k, v in ds.items()}


if do_om2:
    ds_om2 = load_om2()
    print(ds_om2["CT"].time.values[[0, -1]])

## Compute the transformation

As in the COSIMA recipe: salt fluxes are in (kg of salt) m$^{-2}$ s$^{-1}$ and $\beta$ is in kg (g of salt)$^{-1}$, so the salt fluxes are multiplied by 1000;
the water flux is multiplied by surface absolute salinity to convert it to (g of salt) m$^{-2}$ s$^{-1}$.
The heat transformation is divided by $c_p$.

In [ ]:
def compute_salt_transformation(ds):
    haline_contraction = gsw.beta(ds['SA'], ds['CT'], ds['pressure'])
    return haline_contraction * (ds['SA'] * ds['wfo'] - ds['salt_flux'] * 1000)


def compute_heat_transformation(ds):
    thermal_expansion = gsw.alpha(ds['SA'], ds['CT'], ds['pressure'])
    return thermal_expansion * ds['heat_flux'] / c_p

The recipe loops over density bins, sums the (days-per-month weighted) transformation of cells in each bin over time,
divides by the total number of days and by the bin width, and flips the sign so that positive means conversion to denser water.
The function below does exactly that, but bins all cells at once with `np.bincount` rather than looping over bins,
which is much faster at 25 km and finer.

In [ ]:
def isopycnal_bins(ds, transformation, bin_edges=isopycnal_bin_edges):
    """Time-mean transformation (m s-1) per sigma_1 bin at each grid cell; positive = conversion to denser water"""
    sigma = ds['pot_rho_1'].transpose('time', 'y', 'x').values
    tr = (transformation * ds['days_per_month']).transpose('time', 'y', 'x').values
    nt, ny, nx = sigma.shape
    nbins = len(bin_edges) - 1

    # bin i holds bin_edges[i] < sigma <= bin_edges[i+1], as in the recipe
    idx = np.digitize(sigma, bin_edges, right=True) - 1
    valid = (idx >= 0) & (idx < nbins) & np.isfinite(tr)
    cell = np.broadcast_to(np.arange(ny * nx).reshape(1, ny, nx), sigma.shape)
    summed = np.bincount((idx * ny * nx + cell)[valid], weights=tr[valid], minlength=nbins * ny * nx)
    summed = summed.reshape(nbins, ny, nx)

    summed /= float(ds['days_per_month'].sum())
    summed /= np.diff(bin_edges)[:, np.newaxis, np.newaxis]
    summed *= -1

    template = ds['pot_rho_1'].isel(time=0, drop=True)
    return xr.DataArray(
        np.where(summed != 0, summed, np.nan),
        dims=('isopycnal_bins', 'y', 'x'),
        coords={'isopycnal_bins': (bin_edges[1:] + bin_edges[:-1]) / 2, 'y': template.y, 'x': template.x},
    )


def swmt(ds):
    salt_tr = isopycnal_bins(ds, compute_salt_transformation(ds)).rename('salt_transformation')
    heat_tr = isopycnal_bins(ds, compute_heat_transformation(ds)).rename('heat_transformation')
    return xr.Dataset({'net': salt_tr.fillna(0) + heat_tr.fillna(0), 'heat': heat_tr, 'salt': salt_tr})


models = {'ACCESS-OM3': ds_om3}
if do_om2:
    models['ACCESS-OM2'] = ds_om2

tr = {name: swmt(ds) for name, ds in models.items()}

## Antarctic shelf mask

The COSIMA recipe uses a precomputed mask poleward of a continuous approximation of the 1000 m isobath, which only exists for some grids.
Here a mask with the same meaning is built for each grid from its bathymetry:
1. the *open ocean* is the deep (≥ 1000 m) ocean connected to the northern edge of the region;
2. the *shelf* is all other ocean connected to the Antarctic continent, so deep troughs and depressions on the shelf are included,
   while seamounts, banks and sub-Antarctic island shelves are excluded.

Connectivity is periodic in longitude.

In [ ]:
def label_periodic(mask):
    """ndimage.label, joining regions across the zonal periodic boundary"""
    lab, n = ndimage.label(mask)
    parent = np.arange(n + 1)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for a, b in zip(lab[:, 0], lab[:, -1]):
        if a and b:
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra
    roots = np.array([find(i) for i in range(n + 1)])
    return roots[lab]


def antarctic_shelf_mask(depth, isobath=shelf_isobath):
    """True on the Antarctic continental shelf, poleward of the given isobath. depth has dims (y, x), y increasing northward."""
    d = depth.transpose('y', 'x').values
    ocean = np.isfinite(d) & (d > 0)

    deep = label_periodic(ocean & (d >= isobath))
    north = np.unique(deep[-1, :])
    open_ocean = np.isin(deep, north[north > 0])

    land = label_periodic(~ocean)
    south = np.unique(land[0, :])
    antarctica = np.isin(land, south[south > 0])
    next_to_antarctica = ndimage.binary_dilation(antarctica, structure=np.ones((3, 3)))
    next_to_antarctica |= np.roll(antarctica, 1, axis=1) | np.roll(antarctica, -1, axis=1)  # periodic neighbours

    candidates = label_periodic(ocean & ~open_ocean)
    keep = np.unique(candidates[next_to_antarctica & (candidates > 0)])
    shelf = np.isin(candidates, keep[keep > 0])
    return xr.DataArray(shelf, dims=('y', 'x'), coords={'y': depth.y, 'x': depth.x})


shelf = {name: antarctic_shelf_mask(ds['maximum_depth']) for name, ds in models.items()}

In [ ]:
def circular_boundary(ax):
    theta = np.linspace(0, 2 * np.pi, 100)
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    ax.set_boundary(mpath.Path(verts * 0.5 + 0.5), transform=ax.transAxes)


def south_polar_axes(ncols, figsize):
    fig, axes = plt.subplots(ncols=ncols, figsize=figsize, squeeze=False,
                             subplot_kw=dict(projection=ccrs.SouthPolarStereo()))
    axes = axes.ravel()
    for ax in axes:
        ax.set_extent([-180, 180, -90, lat_max], ccrs.PlateCarree())
        circular_boundary(ax)
    return fig, axes


def land_mask(ds):
    return np.isnan(ds['maximum_depth']) | (ds['maximum_depth'] <= 0)

In [ ]:
fig, axes = south_polar_axes(len(models), (6 * len(models), 6))
for i, (ax, (name, ds)) in enumerate(zip(axes, models.items())):
    ax.contourf(ds['maximum_depth'].x, ds['maximum_depth'].y, shelf[name].where(shelf[name]),
                colors='red', alpha=0.5, transform=ccrs.PlateCarree())
    ax.contour(ds['maximum_depth'].x, ds['maximum_depth'].y, ds['maximum_depth'].fillna(0),
               levels=[shelf_isobath], colors='k', linewidths=0.5, transform=ccrs.PlateCarree())
    ax.contourf(ds['maximum_depth'].x, ds['maximum_depth'].y, land_mask(ds).where(land_mask(ds)),
                colors='darkgrey', transform=ccrs.PlateCarree())
    ax.set_title(f"{chr(97 + i)}) {name}")
plt.show()

## Area-integrated transformation (Sv)

In [ ]:
def integrate(name, mask=None):
    t = tr[name] * models[name]['area'] / 1e6
    if mask is not None:
        t = t.where(mask)
    out = t.sum(['x', 'y'])
    for v in out:
        out[v].attrs['units'] = 'Sv'
    return out


swmt_so = {name: integrate(name) for name in models}
swmt_shelf = {name: integrate(name, shelf[name]) for name in models}

In [ ]:
plt.rcParams['font.size'] = 12
colours = {'net': 'k', 'heat': 'tab:red', 'salt': 'tab:blue'}
linestyles = {'ACCESS-OM3': '-', 'ACCESS-OM2': '--'}

fig, axes = plt.subplots(ncols=2, figsize=(13, 7), sharey=True)
for ax, (title, data) in zip(axes, [('a) Southern Ocean south of 59°S', swmt_so),
                                    ('b) Antarctic continental shelf', swmt_shelf)]):
    for name, s in data.items():
        for term, c in colours.items():
            ax.plot(s[term], s.isopycnal_bins, color=c, ls=linestyles[name], label=f'{name} {term}')
    ax.axvline(0, color='k', lw=0.5)
    ax.set_title(title, loc='left', fontweight='bold')
    ax.set_xlabel('Surface water mass transformation (Sv)')
axes[0].set_ylim(33.2, 31)
axes[0].set_ylabel(r'$\sigma_1$ (kg m$^{-3}$)')
axes[1].set_ylim(33.2, 32)
axes[1].yaxis.set_tick_params(labelleft=True)
axes[0].legend(loc='lower left', fontsize=10)
fig.text(0.99, 0.0, f'{t_slice.start}-{t_slice.stop} mean', ha='right', fontsize=10)

mkmd.savefig(fig, "Surface water mass transformation, Southern Ocean and Antarctic shelf",
             f"Surface water mass transformation (Sv) in $\\sigma_1$ classes due to net (black), heat (red) and salt/freshwater (blue) "
             f"surface fluxes, (a) south of 59°S and (b) on the Antarctic continental shelf (poleward of the {shelf_isobath} m isobath), "
             f"averaged over {t_slice.start}-{t_slice.stop}. Solid: ACCESS-OM3"
             + ("; dashed: ACCESS-OM2 (025deg_jra55_iaf_omip2_cycle1)." if do_om2 else ".")
             + " Positive values are transformation to denser water.", dpi=150)
plt.show()

### Where does shelf dense water formation happen?

Map the transformation rate across the density at which the net shelf transformation peaks, separately for each model
(the COSIMA recipe used a fixed $\sigma_1$ = 32.6 kg m$^{-3}$ for its Pan-Antarctic 1/10° run).
The area-integral of these maps over the shelf is the peak shelf transformation in the figure above.

In [ ]:
peak = {}
for name, s in swmt_shelf.items():
    i = int(s['net'].argmax())
    peak[name] = dict(sigma1=float(s.isopycnal_bins[i]), net=float(s['net'][i]),
                      heat=float(s['heat'][i]), salt=float(s['salt'][i]))

table = ["| Model | Peak density $\\sigma_1$ (kg m$^{-3}$) | Net (Sv) | Heat (Sv) | Salt (Sv) |",
         "| --- | --- | --- | --- | --- |"]
table += [f"| {name} | {p['sigma1']:.2f} | {p['net']:.2f} | {p['heat']:.2f} | {p['salt']:.2f} |" for name, p in peak.items()]
print("\n".join(table))
mkmd.table(f"Peak Antarctic shelf surface water mass transformation ({t_slice.start}-{t_slice.stop} mean)", table)

In [ ]:
fig, axes = south_polar_axes(len(models), (7 * len(models), 7.5))
for i, (ax, name) in enumerate(zip(axes, models)):
    ds = models[name]
    m = tr[name]['net'].sel(isopycnal_bins=peak[name]['sigma1']) * 1e5
    pc = ax.pcolormesh(m.x, m.y, m.where(m != 0), vmin=0, vmax=2.5, cmap=cm.cm.matter, transform=ccrs.PlateCarree())
    ax.contour(m.x, m.y, shelf[name].astype(float), levels=[0.5], colors='r', linewidths=0.8, transform=ccrs.PlateCarree())
    ax.contourf(m.x, m.y, land_mask(ds).where(land_mask(ds)), colors='darkgrey', transform=ccrs.PlateCarree())
    ax.set_title(f"{chr(97 + i)}) {name}, $\\sigma_1$ = {peak[name]['sigma1']:.2f} kg m$^{{-3}}$", fontsize=12)
cbar = fig.colorbar(pc, ax=list(axes), orientation='horizontal', shrink=0.4, pad=0.04, extend='max')
cbar.set_label(r'Surface water mass transformation ($\times 10^{-5}$ m s$^{-1}$)')

mkmd.savefig(fig, "Map of Antarctic shelf surface water mass transformation",
             f"Net surface water mass transformation ($\\times 10^{{-5}}$ m s$^{{-1}}$) across the $\\sigma_1$ surface at which net Antarctic "
             f"shelf transformation peaks (given in each panel title), averaged over {t_slice.start}-{t_slice.stop}. "
             f"Red contour: shelf mask (poleward of the {shelf_isobath} m isobath).", dpi=150)
plt.show()

In [ ]:
client.close()